In [37]:
!pip install scikit-learn


[notice] A new release of pip is available: 24.1.2 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [38]:
!pip install tensorflow opencv-python


[notice] A new release of pip is available: 24.1.2 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [39]:
import numpy as np
import os
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.layers import Dense, Conv2D, Dropout, Flatten, MaxPooling2D, BatchNormalization
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, accuracy_score, f1_score, recall_score, precision_score
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
import pandas as pd
import os
import cv2
from sklearn.utils import shuffle
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort
import kagglehub

In [40]:
path = kagglehub.dataset_download("rakshithaprasad204/shar-2")
print("Path to dataset files:", path)

print("Contenido de la carpeta base:", os.listdir(path))

Path to dataset files: C:\Users\valen\.cache\kagglehub\datasets\rakshithaprasad204\shar-2\versions\1
Contenido de la carpeta base: ['SHAR-2']


In [41]:
# Cargar el modelo preentrenado YOLOv8
modelo = YOLO("yolov8n.pt")
tracker = DeepSort(max_age=30)

In [42]:
normal= os.path.join(path, "SHAR-2", "Non-Suspicious")
suspicious = os.path.join(path, "SHAR-2","Suspicious")

videos = []
labelsDict = {0: "Normal", 1: "Suspicious"}
labels = []

for video in os.listdir(normal):
    videos.append(os.path.join(normal, video))
    labels.append(0)

for video in os.listdir(suspicious):
    videos.append(os.path.join(suspicious, video))
    labels.append(1)

dt = pd.DataFrame({"video": videos, "label": labels})


In [45]:
# Diccionario para almacenar los frames de cada persona
personas_tensores = []  # Lista para almacenar los tensores de cada persona
comportamiento = []  # Lista de etiquetas de comportamiento

def detectar_personas(video, label):
    print(f"Procesando video: {video}")
    personas_frames = {}

    # Cargar el video
    cap = cv2.VideoCapture(video)
    if not cap.isOpened():
        print(f"Error al cargar el video: {video}")
        return

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Detección con YOLO
        results = modelo(frame)

        detections = []
        for result in results:
            for box in result.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])  # Coordenadas de la persona
                confidence = box.conf[0].item()
                class_id = int(box.cls[0].item())

                # Filtrar solo personas (clase 0 en COCO)
                if class_id == 0:
                    detections.append([[x1, y1, x2, y2], confidence, "person"])

        # Aplicar seguimiento con DeepSORT
        tracked_objects = tracker.update_tracks(detections, frame=frame)

        for track in tracked_objects:
            if not track.is_confirmed():
                continue

            track_id = track.track_id  # ID único del individuo
            x1, y1, x2, y2 = track.to_tlbr().astype(int)  # Obtener coordenadas del tracking

            # Evitar recortes fuera del tamaño de la imagen
            h, w, _ = frame.shape
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w, x2), min(h, y2)

            # Extraer el recorte de la persona
            persona_recorte = frame[y1:y2, x1:x2]

            # Si el recorte es válido (aún tiene tamaño después de los límites)
            if persona_recorte.size == 0:
                continue

            # Guardar frame en la lista de la persona
            if track_id not in personas_frames:
                personas_frames[track_id] = []
            
            personas_frames[track_id].append(persona_recorte)

    cap.release()

    # Definir la función de procesamiento
    def procesar_frames(frames, target_size=(224, 224)):
        frames_redimensionados = [cv2.resize(frame, target_size) for frame in frames]
        return tf.convert_to_tensor(np.array(frames_redimensionados) / 255.0, dtype=tf.float32)

    for track_id, frames in personas_frames.items():
        if len(frames) < 10:
            continue
        
        video_tensor = procesar_frames(frames)
        personas_tensores.append(video_tensor)
        comportamiento.append(label) 


In [46]:
for video, label in zip(videos, labels):
    detectar_personas(video, label)

print(f"Total de personas detectadas: {len(personas_tensores)}")
print (labels)


Procesando video: C:\Users\valen\.cache\kagglehub\datasets\rakshithaprasad204\shar-2\versions\1\SHAR-2\Non-Suspicious\1.mpg



0: 448x640 13 persons, 1 truck, 179.7ms
Speed: 4.3ms preprocess, 179.7ms inference, 1.0ms postprocess per image at shape (1, 3, 448, 640)

0: 448x640 12 persons, 55.1ms
Speed: 2.0ms preprocess, 55.1ms inference, 1.0ms postprocess per image at shape (1, 3, 448, 640)

0: 448x640 11 persons, 1 truck, 1 handbag, 66.8ms
Speed: 2.0ms preprocess, 66.8ms inference, 1.0ms postprocess per image at shape (1, 3, 448, 640)

0: 448x640 9 persons, 1 truck, 61.6ms
Speed: 1.0ms preprocess, 61.6ms inference, 2.0ms postprocess per image at shape (1, 3, 448, 640)

0: 448x640 12 persons, 1 handbag, 58.7ms
Speed: 1.2ms preprocess, 58.7ms inference, 0.0ms postprocess per image at shape (1, 3, 448, 640)

0: 448x640 8 persons, 1 truck, 1 handbag, 61.4ms
Speed: 1.0ms preprocess, 61.4ms inference, 1.0ms postprocess per image at shape (1, 3, 448, 640)

0: 448x640 13 persons, 1 handbag, 59.7ms
Speed: 2.0ms preprocess, 59.7ms inference, 1.0ms postprocess per image at shape (1, 3, 448, 640)

0: 448x640 10 persons, 1

In [48]:
# Convertir la lista de tensores en una estructura compatible con TensorFlow
personas_tensores = tf.ragged.constant(personas_tensores, dtype=tf.float32)

# Crear dataset con un generador en lugar de cargar todo en memoria
def generar_datos():
    for video, label in zip(personas_tensores, comportamiento):
        yield video, label

dataset = tf.data.Dataset.from_generator(
    generar_datos,
    output_signature=(
        tf.TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32),  # Secuencia de frames
        tf.TensorSpec(shape=(), dtype=tf.int32)  # Etiqueta
    )
)

# Mezclar y dividir en lotes
dataset = dataset.shuffle(len(personas_tensores)).batch(4).prefetch(tf.data.AUTOTUNE)

KeyboardInterrupt: 

In [ ]:
# Definir proporción de entrenamiento (80% train, 20% validación)
train_size = int(0.8 * len(personas_tensores))

# Crear datasets de entrenamiento y validación
train_dataset = dataset.take(train_size)
val_dataset = dataset.skip(train_size) 

In [ ]:
# Crear un modelo 3D CNN
def create_3d_cnn(input_shape, num_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.Conv3D(32, kernel_size=(3, 3, 3), activation='relu', input_shape=input_shape),
        tf.keras.layers.MaxPooling3D(pool_size=(2, 2, 2)),
        tf.keras.layers.Conv3D(64, kernel_size=(3, 3, 3), activation='relu'),
        tf.keras.layers.MaxPooling3D(pool_size=(2, 2, 2)),
        tf.keras.layers.Conv3D(128, kernel_size=(3, 3, 3), activation='relu'),
        tf.keras.layers.MaxPooling3D(pool_size=(2, 2, 2)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    return model

# Crear el modelo
input_shape = (10, 224, 224, 3)
num_classes = len(np.unique(labels))
model = create_3d_cnn(input_shape, num_classes)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
#earlyStopping = EarlyStopping(monitor='val_accuracy', patience=5, verbose=0, restore_best_weights=True)

# Entrenamiento del modelo
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=1,
    batch_size=16,
    #callbacks=[earlyStopping]
)

In [ ]:
# Evaluar el modelo
loss, accuracy = model.evaluate(X_val, y_val)
print(f"Pérdida: {loss}, Precisión: {accuracy}")

In [ ]:
# reporte de clasificación
y_pred = model.predict(X_val)
y_pred = np.argmax(y_pred, axis=1)
print(classification_report(y_val, y_pred))

# Matriz de confusión
cm = confusion_matrix(y_val, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.unique(labels))
disp.plot()

# Guardar el modelo
model.save('modelSuspicious.keras')
